# Rôle 1 : modèles de rentabilité et de risque

1. Préparation : jeux, rentabilités aberrantes, mesures
2. Modèles de référence (tâche 5) : `DummyRegressor`, `LinearRegression`, `DummyClassifier`

Règles suivies dans tout le notebook :

- les modèles sont appris sur le **jeu d'entraînement** (1 050 projets) et comparés sur le **jeu de validation** (225 projets) ;
- le prétraitement (`supervise.creer_pretraitement`) est placé dans un `Pipeline` avec le modèle : il n'apprend que sur l'entraînement ;
- toute règle de nettoyage est **apprise sur l'entraînement seulement**, puis appliquée telle quelle à la validation ;
- le **jeu de test n'est pas utilisé ici** : il sera ouvert une seule fois, à la fin, pour le modèle choisi (tâche 8).

Rappel : `rentabilite` et `risque` sont **simulées** (voir `data/README.md`) ; le bruit de la simulation empêche tout modèle d'être parfait. Le contrôle du script de préparation donne un R² d'environ 0,55 et 58 % de bonnes classes de risque avec un modèle linéaire.

In [1]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline

RACINE = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(RACINE))
from src import supervise

logging.basicConfig(level=logging.INFO, format="%(message)s")
GRAINE = 42
pd.set_option("display.width", 120)

## 1. Préparation

In [2]:
jeux = supervise.charger_jeux()
entrainement, validation = jeux["entrainement"], jeux["validation"]

Jeux chargés : {'entrainement': 1050, 'validation': 225, 'test': 225}


### 1.1 Rentabilités aberrantes : une règle apprise sur l'entraînement

L'analyse exploratoire a montré quelques rentabilités très loin des autres (au-delà de ±35 %, alors que la moitié des projets est entre 6 et 11 %). Une régression par moindres carrés élève les erreurs au carré : quelques points de ce genre peuvent tirer tout le modèle.

On ne peut pas savoir d'avance quels projets sont faux. On utilise donc une **règle statistique classique**, apprise sur l'entraînement seulement :

- Q1 et Q3 sont les quartiles de la rentabilité (25 % et 75 % des projets en dessous) ; IQR = Q3 − Q1 est l'écart interquartile ;
- une rentabilité est **aberrante** si elle sort de **[Q1 − 3·IQR ; Q3 + 3·IQR]**.

Le facteur 3 (au lieu du 1,5 des boîtes à moustaches) ne vise que les valeurs **très** éloignées. Les quartiles ne bougent presque pas à cause de quelques valeurs extrêmes : la règle n'est pas faussée par ce qu'elle doit repérer.

In [3]:
bornes = supervise.bornes_rentabilite(entrainement[supervise.CIBLE_REGRESSION])
print(f"Bornes apprises sur l'entraînement : [{bornes[0]:.1f} % ; {bornes[1]:.1f} %]")

hors_bornes = pd.DataFrame({
    nom: [len(jeu), (~supervise.dans_les_bornes(jeu, bornes)).sum()]
    for nom, jeu in jeux.items()
}, index=["projets", "rentabilité hors bornes"])
hors_bornes

Bornes apprises sur l'entraînement : [-6.1 % ; 23.3 %]


,entrainement,validation,test
projets,1050,225,225
rentabilité hors bornes,6,2,2


Les **mêmes bornes** (apprises sur l'entraînement) sont appliquées à la validation et au test : on ne recalcule rien sur ces jeux.

**Vérification après coup.** Le fichier `anomalies_injectees.csv` donne la liste des anomalies fabriquées. Il ne sert **ni à apprendre ni à filtrer** : seulement à contrôler, une fois la règle appliquée, ce qu'elle a attrapé.

In [4]:
anomalies = pd.read_csv(RACINE / "data" / "processed" / "anomalies_injectees.csv")
controle = pd.concat([jeu.assign(jeu=nom) for nom, jeu in jeux.items()])
controle["hors bornes"] = ~supervise.dans_les_bornes(controle, bornes)
controle = controle.merge(anomalies[["id_projet", "type_anomalie"]], on="id_projet", how="left")
controle["type_anomalie"] = controle["type_anomalie"].fillna("aucune")
pd.crosstab(controle["type_anomalie"], controle["hors bornes"], margins=True)

hors bornes,False,True,All
type_anomalie,,,
aucune,1475,0,1475
cout_aberrant,7,0,7
duree_incoherente,8,0,8
rentabilite_extreme,0,10,10
All,1490,10,1500


La règle attrape les **10 rentabilités fabriquées** (`rentabilite_extreme`) des projets terminés, et **aucun projet normal** ni aucune autre anomalie. C'est logique : les anomalies de coût ou de durée faussent une **entrée**, pas la rentabilité, donc une règle sur la rentabilité ne peut pas les voir. Ces projets restent dans l'entraînement ; le passage du coût au log10 atténue déjà les coûts multipliés par 10 à 50. Leur repérage est le travail du rôle 2.

Le risque n'est pas touché par ces anomalies : il a été calculé avant leur injection.

### 1.2 Mesures

In [5]:
validation_dans_bornes = validation[supervise.dans_les_bornes(validation, bornes)]
jeux_evaluation = {"validation complète": validation, "validation dans les bornes": validation_dans_bornes}
print({nom: len(jeu) for nom, jeu in jeux_evaluation.items()})


def pipeline(modele) -> Pipeline:
    """Prétraitement commun suivi du modèle : tout s'ajuste sur les données passées à fit."""
    return Pipeline([("pretraitement", supervise.creer_pretraitement()), ("modele", modele)])


def mesures_regression(y_vrai, y_predit) -> dict:
    """RMSE, MAE (en points de rentabilité) et R²."""
    return {"RMSE": root_mean_squared_error(y_vrai, y_predit),
            "MAE": mean_absolute_error(y_vrai, y_predit),
            "R2": r2_score(y_vrai, y_predit)}


def mesures_classification(y_vrai, y_predit) -> dict:
    """Exactitude et F1 macro (moyenne des F1 des trois classes, chacune avec le même poids)."""
    return {"exactitude": accuracy_score(y_vrai, y_predit),
            "F1 macro": f1_score(y_vrai, y_predit, average="macro")}

{'validation complète': 225, 'validation dans les bornes': 223}


Les mesures de régression sont calculées sur deux versions de la validation :

- **validation complète** (225 projets), telle qu'elle arrive ;
- **validation dans les bornes** (223 projets) : sans les rentabilités que la règle apprise sur l'entraînement juge aberrantes.

**Les mesures.**

- **RMSE** (racine de l'erreur quadratique moyenne) : l'erreur typique, en points de rentabilité. Elle élève les erreurs au carré, donc punit fort les grosses erreurs.
- **MAE** (erreur absolue moyenne) : l'erreur moyenne en points, moins sensible aux valeurs extrêmes.
- **R²** : part de la variance de la rentabilité expliquée par le modèle. 0 = pas mieux que prédire la moyenne ; 1 = parfait ; négatif = pire que la moyenne.
- **Exactitude** : part de projets bien classés.
- **F1 macro** : pour chaque classe, le F1 combine précision et rappel ; on fait la moyenne des trois classes sans pondérer par leur taille. Un modèle qui ignore la classe rare `eleve` est donc pénalisé, alors que son exactitude peut rester correcte.

## 2. Modèles de référence (tâche 5)

Un modèle de référence donne le niveau à battre. Sans lui, un R² ou une exactitude ne veulent rien dire.

- `DummyRegressor` : prédit toujours la **moyenne** de la rentabilité d'entraînement ;
- `LinearRegression` : régression linéaire par moindres carrés, sans régularisation ;
- `DummyClassifier` : répond toujours la **classe la plus fréquente** (`most_frequent`), ou tire une classe au hasard selon les proportions d'entraînement (`stratified`).

### 2.1 Régression de la rentabilité

On apprend chaque modèle sur deux versions de l'entraînement, pour mesurer l'effet des rentabilités aberrantes :

- **tout l'entraînement** (1 050 projets) ;
- **entraînement dans les bornes** (sans les rentabilités hors de la règle).

In [6]:
versions_entrainement = {
    "tout l'entraînement": entrainement,
    "entraînement dans les bornes": entrainement[supervise.dans_les_bornes(entrainement, bornes)],
}
modeles_reference = {"DummyRegressor (moyenne)": DummyRegressor(strategy="mean"),
                     "LinearRegression": LinearRegression()}

lignes = []
for nom_entrainement, donnees in versions_entrainement.items():
    for nom_modele, modele in modeles_reference.items():
        ajuste = pipeline(modele).fit(donnees[supervise.VARIABLES], donnees[supervise.CIBLE_REGRESSION])
        ligne = {"modèle": nom_modele, "entraînement": nom_entrainement, "projets appris": len(donnees)}
        ligne["R2 entraînement"] = r2_score(donnees[supervise.CIBLE_REGRESSION],
                                            ajuste.predict(donnees[supervise.VARIABLES]))
        for nom_evaluation, evaluation in jeux_evaluation.items():
            predit = ajuste.predict(evaluation[supervise.VARIABLES])
            for mesure, valeur in mesures_regression(evaluation[supervise.CIBLE_REGRESSION], predit).items():
                ligne[f"{mesure} ({nom_evaluation})"] = valeur
        lignes.append(ligne)

resultats_reference = pd.DataFrame(lignes)
resultats_reference.round(3)

,modèle,entraînement,projets appris,R2 entraînement,RMSE (validation complète),MAE (validation complète),R2 (validation complète),RMSE (validation dans les bornes),MAE (validation dans les bornes),R2 (validation dans les bornes)
0,DummyRegressor (moyenne),tout l'entraînement,1050,0.000,5.496,2.660,-0.021,2.803,2.262,-0.053
1,LinearRegression,tout l'entraînement,1050,0.405,5.286,2.210,0.056,2.413,1.816,0.219
2,DummyRegressor (moyenne),entraînement dans les bornes,1044,0.000,5.478,2.639,-0.014,2.775,2.240,-0.033
3,LinearRegression,entraînement dans les bornes,1044,0.665,5.230,2.035,0.076,2.142,1.632,0.385


In [7]:
colonnes = ["modèle", "entraînement", "R2 entraînement",
            "RMSE (validation complète)", "R2 (validation complète)",
            "RMSE (validation dans les bornes)", "MAE (validation dans les bornes)",
            "R2 (validation dans les bornes)"]
resultats_reference[colonnes].round(3)

,modèle,entraînement,R2 entraînement,RMSE (validation complète),R2 (validation complète),RMSE (validation dans les bornes),MAE (validation dans les bornes),R2 (validation dans les bornes)
0,DummyRegressor (moyenne),tout l'entraînement,0.000,5.496,-0.021,2.803,2.262,-0.053
1,LinearRegression,tout l'entraînement,0.405,5.286,0.056,2.413,1.816,0.219
2,DummyRegressor (moyenne),entraînement dans les bornes,0.000,5.478,-0.014,2.775,2.240,-0.033
3,LinearRegression,entraînement dans les bornes,0.665,5.230,0.076,2.142,1.632,0.385


Ce que montre le tableau :

1. **Le modèle « moyenne » a un R² proche de 0**, par construction : c'est le plancher. Son erreur typique est d'environ 2,8 points sur la validation dans les bornes.
2. **Les rentabilités aberrantes de l'entraînement faussent l'apprentissage** : les retirer fait passer le R² de la régression linéaire d'environ 0,22 à 0,39 sur la validation dans les bornes, et sa MAE de 1,8 à 1,6 point.
3. **Les 2 rentabilités aberrantes de la validation écrasent les mesures** : avec elles, la RMSE fait plus que doubler (de 2,1 à 5,2 points) et le R² tombe sous 0,1, quel que soit le modèle. Ces deux projets représentent 1 % de la validation mais l'essentiel de l'erreur au carré. On donne les deux mesures : la validation complète dit ce qui se passe sur des données telles qu'elles arrivent, la validation dans les bornes dit ce que le modèle sait vraiment prédire.
4. **La régression linéaire apprend trop les données d'entraînement** : R² de 0,67 sur l'entraînement contre 0,39 en validation. Avec 175 colonnes pour un millier de projets, dont 158 pays souvent représentés par un ou deux projets, elle donne à chaque pays rare un coefficient qui colle au bruit de ces quelques projets. C'est exactement ce que la régularisation (Ridge, Lasso, ElasticNet, tâche 6) doit corriger, et ce qui explique qu'on reste sous le plafond d'environ 0,55 annoncé par le script de préparation.

### 2.2 Classification du risque

In [8]:
lignes = []
for strategie in ["most_frequent", "stratified"]:
    ajuste = pipeline(DummyClassifier(strategy=strategie, random_state=GRAINE)).fit(
        entrainement[supervise.VARIABLES], entrainement[supervise.CIBLE_CLASSIFICATION])
    predit = ajuste.predict(validation[supervise.VARIABLES])
    lignes.append({"modèle": f"DummyClassifier ({strategie})",
                   **mesures_classification(validation[supervise.CIBLE_CLASSIFICATION], predit)})
resultats_reference_risque = pd.DataFrame(lignes)
resultats_reference_risque.round(3)

,modèle,exactitude,F1 macro
0,DummyClassifier (most_frequent),0.400,0.190
1,DummyClassifier (stratified),0.356,0.333


- **Toujours la classe la plus fréquente** : 40 % d'exactitude (la classe `faible`, à égalité avec `moyen`, représente 40 % des projets), mais un F1 macro de 0,19, car deux classes sur trois ne sont jamais prédites.
- **Tirage au hasard selon les proportions** : exactitude d'environ 36 % et F1 macro d'environ 0,33 ; il prédit toutes les classes, mais au hasard.

Ce sont les niveaux à battre : un classifieur utile doit dépasser **40 % d'exactitude** et un **F1 macro de 0,33**. Le contrôle du script de préparation annonce environ 58 % pour un modèle linéaire.

Pour le risque, on garde tous les projets, en entraînement comme en validation.

### 2.3 Bilan des modèles de référence

| Tâche | Référence | Validation complète | Validation dans les bornes |
|---|---|---|---|
| Rentabilité | moyenne d'entraînement | R² ≈ 0, RMSE ≈ 5,5 points | R² ≈ 0, RMSE ≈ 2,8 points |
| Rentabilité | régression linéaire, entraînement dans les bornes | R² ≈ 0,08, RMSE ≈ 5,2 points | R² ≈ 0,39, RMSE ≈ 2,1 points |
| Risque | classe la plus fréquente | exactitude 40 %, F1 macro 0,19 | (identique) |
| Risque | tirage selon les proportions | exactitude ≈ 36 %, F1 macro ≈ 0,33 | (identique) |

Choix pour la suite :

- les modèles de rentabilité sont appris sur l'**entraînement dans les bornes** (règle Q1 − 3·IQR ; Q3 + 3·IQR apprise sur l'entraînement) ;
- ils sont mesurés sur la **validation complète** et sur la **validation dans les bornes** (mêmes bornes) ; le test sera traité de la même façon.